# Customer Churn Prediction Using Ensemble Machine Learning

**Datheon Machine Learning & Data Science Internship — Task 2**

**Objective:** Predict telecom customer churn using Random Forest and XGBoost, with preprocessing, class-imbalance handling, cross-validation, ROC-AUC evaluation, confusion matrices, and top-10 feature importances.

> Run all cells in order. Results are generated from the real dataset; no metrics have been pre-filled or fabricated.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, accuracy_score)
from xgboost import XGBClassifier

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 1. Load dataset

The notebook first checks for a local CSV, then tries the dataset URL provided in the internship task. If the download fails, manually download the CSV from the link in the README and place it next to this notebook.

In [ ]:
dataset_file = Path("Telco-Customer-Churn.csv")
dataset_url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

if dataset_file.exists():
    df = pd.read_csv(dataset_file)
else:
    try:
        df = pd.read_csv(dataset_url)
        df.to_csv(dataset_file, index=False)
    except Exception as exc:
        raise FileNotFoundError(
            "Could not download the dataset. Download the CSV manually from the "
            "dataset URL in README.md and save it as Telco-Customer-Churn.csv "
            "in the notebook folder."
        ) from exc

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())

## 2. Clean and inspect data

In [ ]:
df = df.copy()
df.columns = df.columns.str.strip()
assert "Churn" in df.columns, "Expected a Churn column in the dataset."

if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df = df.drop(columns=["customerID"], errors="ignore")
df = df.drop_duplicates()
df = df.dropna(subset=["Churn"])
y = df["Churn"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
if y.isna().any():
    raise ValueError("Unexpected values in Churn target.")
X = df.drop(columns=["Churn"])
print("Missing values (top 10):")
display(X.isna().sum().sort_values(ascending=False).head(10))
print("Target distribution:")
display(y.value_counts().rename(index={0:"No churn",1:"Churn"}))
sns.countplot(x=y.map({0:"No churn",1:"Churn"}))
plt.title("Target class distribution")
plt.show()

## 3. Stratified train/test split and preprocessing

The test set remains untouched during model selection. `class_weight` and `scale_pos_weight` are applied only in training, avoiding SMOTE leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.columns.difference(numeric_cols).tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Numerical features:", numeric_cols)
print("Categorical features:", categorical_cols)

## 4. Random Forest and XGBoost with 5-fold stratified cross-validation

In [ ]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
class_ratio = negative_count / positive_count

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=250, max_depth=9, min_samples_leaf=3,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        objective="binary:logistic", eval_metric="logloss",
        scale_pos_weight=class_ratio, random_state=RANDOM_STATE,
        n_jobs=2, tree_method="hist"
    )
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
trained = {}
results = []
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring={"auc": "roc_auc", "f1": "f1"},
        n_jobs=1
    )
    pipe.fit(X_train, y_train)
    probabilities = pipe.predict_proba(X_test)[:, 1]
    predictions = pipe.predict(X_test)
    trained[name] = pipe
    results.append({
        "Model": name,
        "CV ROC-AUC mean": scores["test_auc"].mean(),
        "CV ROC-AUC std": scores["test_auc"].std(),
        "CV F1 mean": scores["test_f1"].mean(),
        "Test ROC-AUC": roc_auc_score(y_test, probabilities),
        "Test accuracy": accuracy_score(y_test, predictions)
    })
results_df = pd.DataFrame(results).sort_values("Test ROC-AUC", ascending=False)
display(results_df.round(4))
print("Target: ROC-AUC > 0.80")
for _, item in results_df.iterrows():
    print(f'{item["Model"]}: {"TARGET MET" if item["Test ROC-AUC"] > 0.80 else "TARGET NOT MET"}')

## 5. ROC-AUC curves

In [ ]:
plt.figure(figsize=(8, 6))
for name, pipe in trained.items():
    scores = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random classifier")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves on held-out test data")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=180)
plt.show()

## 6. Confusion matrices and classification reports

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, pipe) in zip(axes, trained.items()):
    pred = pipe.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["No churn", "Churn"],
        cmap="Blues", ax=ax, colorbar=False
    )
    ax.set_title(name)
    print(f"\n{name}\n")
    print(classification_report(y_test, pred, target_names=["No churn", "Churn"]))
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=180)
plt.show()

## 7. Top 10 feature importances for both models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (name, pipe) in zip(axes, trained.items()):
    feature_names = pipe.named_steps["preprocess"].get_feature_names_out()
    importances = pipe.named_steps["model"].feature_importances_
    top = pd.Series(importances, index=feature_names).nlargest(10).sort_values()
    top.plot(kind="barh", ax=ax)
    ax.set_title(f"Top 10 features: {name}")
    ax.set_xlabel("Model feature importance")
plt.tight_layout()
plt.savefig("feature_importance_top10.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Conclusion

After running the notebook, report the actual cross-validation ROC-AUC, test ROC-AUC, confusion matrices, and top features. The >0.80 ROC-AUC target is a goal, **not a guaranteed outcome**. Note that model feature importances describe predictive influence, not causation.

In [ ]:
print("FINAL TEST RESULTS")
display(results_df.round(4))
best = results_df.iloc[0]
print(f'Best test ROC-AUC: {best["Model"]} = {best["Test ROC-AUC"]:.4f}')
print("Saved figures: roc_curves.png, confusion_matrices.png, feature_importance_top10.png")